##### deal with data leakage

In [1]:
import warnings
import json
import os
import numpy as np
import pandas as pd
from datetime import datetime
from typing import Tuple, Dict, List

# Sklearn imports
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, cross_val_score, train_test_split)
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, roc_auc_score,confusion_matrix, average_precision_score,precision_recall_curve, roc_curve)
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek  # optional: cleans borderline samples too

warnings.filterwarnings("ignore")
np.random.seed(42)

In [2]:
df = pd.read_csv(r'D:\MajorProject\EDA\hormones_clean.csv')
df.columns

Index(['id', 'study_interval', 'is_weekend', 'day_in_study', 'phase', 'lh',
       'estrogen', 'pdg', 'flow_volume', 'flow_color', 'appetite',
       'exerciselevel', 'headaches', 'cramps', 'sorebreasts', 'fatigue',
       'sleepissue', 'moodswing', 'stress', 'foodcravings', 'indigestion',
       'bloating', 'is_bleeding', 'flow_grp_Abnormal', 'flow_grp_Fresh',
       'flow_grp_No Flow', 'flow_grp_Old', 'flow_volume_numeric',
       'period_start', 'period_id', 'day_in_period', 'flow_volume_imputed',
       'lh_surge', 'pdg_imputed', 'estrogen_capped_flag', 'estrogen_clean',
       'estrogen_imputed', 'high_estrogen_flag', 'lh_imputed',
       'cramps_imputed', 'sorebreasts_imputed', 'bloating_imputed',
       'moodswing_imputed', 'fatigue_imputed', 'headaches_imputed',
       'foodcravings_imputed', 'indigestion_imputed', 'exerciselevel_imputed',
       'stress_imputed', 'sleepissue_imputed', 'appetite_imputed'],
      dtype='object')

In [3]:
# SECTION 1: CONFIGURATION

TARGET_PERIOD   = "period_start"
TARGET_OVUL     = "lh_surge"

# Features selected based on clinical relevance + EDA
# Grouped by domain for interpretability
HORMONAL_FEATURES = [
    "lh_imputed", "estrogen_imputed", "pdg_imputed",
    "high_estrogen_flag", "estrogen_capped_flag"
]

CYCLE_FEATURES = [
    # "day_in_period", 
    "day_in_study", 
    # "phase",
    # "study_interval", 
    "is_weekend", 
    # "period_id"
]

SYMPTOM_FEATURES = [
    "cramps_imputed", "sorebreasts_imputed", "bloating_imputed",
    "moodswing_imputed", "fatigue_imputed", "headaches_imputed",
    "foodcravings_imputed", "indigestion_imputed",
    "exerciselevel_imputed", "stress_imputed",
    "sleepissue_imputed", "appetite_imputed"
]

# FLOW_FEATURES = [
#     "flow_volume_imputed", "is_bleeding",
#     "flow_grp_Abnormal", "flow_grp_Fresh",
#     "flow_grp_No Flow", "flow_grp_Old"
# ]

ALL_FEATURES = HORMONAL_FEATURES + CYCLE_FEATURES + SYMPTOM_FEATURES

In [4]:
# ❌ REMOVE — derived from target
# "period_id"

# ✅ REPLACE WITH — estimated from data patterns only
def add_cycle_proxy_features(df):
    """
    Estimates cycle position WITHOUT using period_start or is_bleeding.
    Uses only hormonal patterns observable in real-time.
    """
    df = df.copy()
    
    # Days since PDG was at its minimum (follicular baseline)
    # PDG is low in follicular phase, rises after ovulation
    df['pdg_above_threshold'] = (df['pdg_imputed'] > 3.0).astype(int)
    
    # Cumulative LH surge count as cycle proxy
    # (only counts confirmed surges, not future ones)
    df['lh_surge_count'] = (
        df.groupby('id')['lh_surge']
          .transform(lambda x: x.shift(1).fillna(0).cumsum())
        if 'lh_surge' in df.columns 
        else 0
    )
    
    # Days since last LH surge (observable, not target-derived)
    # This is safe because it uses lh_surge which is a separate label
    
    return df

In [5]:
add_cycle_proxy_features(df)

,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,flow_color,...,fatigue_imputed,headaches_imputed,foodcravings_imputed,indigestion_imputed,exerciselevel_imputed,stress_imputed,sleepissue_imputed,appetite_imputed,pdg_above_threshold,lh_surge_count
0,1,2022,True,1,2,2.9,94.2,NaN,Not at all,Not at all,...,4.0,4.0,1.0,1.0,3.0,3.0,1.0,3.0,0,0
1,1,2022,False,2,2,1.2,226.3,NaN,Not at all,Not at all,...,4.0,5.0,1.0,1.0,3.0,3.0,5.0,3.0,0,0
2,1,2022,False,3,2,3.5,276.8,NaN,Not at all,Not at all,...,5.0,4.0,1.0,1.0,3.0,1.0,5.0,3.0,0,0
3,1,2022,False,4,3,1.8,322.1,NaN,Not at all,Not at all,...,4.0,1.0,1.0,1.0,3.0,1.0,5.0,3.0,0,0
4,1,2022,False,5,3,4.6,244.9,NaN,Not at all,Not at all,...,4.0,1.0,1.0,1.0,3.0,1.0,4.0,3.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5654,50,2024,False,947,4,4.6,70.7,2.8,NaN,NaN,...,3.0,1.0,1.0,1.0,3.0,3.0,3.0,4.0,0,22
5655,50,2024,False,948,4,5.8,87.0,7.0,NaN,NaN,...,3.0,1.0,1.0,1.0,3.0,3.0,3.0,4.0,1,22
5656,50,2024,False,949,4,NaN,NaN,NaN,NaN,NaN,...,3.0,1.0,1.0,1.0,3.0,3.0,3.0,4.0,1,22
5657,50,2024,False,950,1,NaN,NaN,NaN,NaN,NaN,...,3.0,1.0,1.0,1.0,3.0,3.0,3.0,4.0,0,22


In [6]:
# use instead of prepare data
def temporal_train_test_split(df, target, feature_cols, 
                               test_frac=0.2, min_train_days=60):
    """
    Proper temporal split: train on past, test on future.
    Respects the causal direction of time — no future data 
    bleeds into training features.
    """
    df = df.sort_values(['id', 'day_in_study']).copy()
    
    # Per-participant cutoff: last 20% of each person's days = test
    train_rows, test_rows = [], []
    
    for pid, group in df.groupby('id'):
        n = len(group)
        if n < min_train_days:
            train_rows.append(group)  # too short → all training
            continue
        cutoff = int(n * (1 - test_frac))
        train_rows.append(group.iloc[:cutoff])
        test_rows.append(group.iloc[cutoff:])
    
    train_df = pd.concat(train_rows)
    if not test_rows:
        raise ValueError(
            "Temporal split produced an empty test set. "
            "Lower min_train_days or increase participant history."
        )
    test_df  = pd.concat(test_rows)
    
    available = [f for f in feature_cols if f in df.columns]
    
    X_train = train_df[available].fillna(0)
    X_test  = test_df[available].fillna(0)
    y_train = train_df[target].astype(int)
    y_test  = test_df[target].astype(int)
    
    print(f"\n[Temporal Split] Target: '{target}'")
    print(f"  Train: {len(X_train)} rows | "
          f"Positive rate: {y_train.mean():.3f}")
    print(f"  Test:  {len(X_test)} rows  | "
          f"Positive rate: {y_test.mean():.3f}")
    print(f"  Test covers FUTURE days only — no temporal leakage")
    
    return X_train, X_test, y_train, y_test, available

In [7]:
# =============================================================================
# SECTION 2: FEATURE ENGINEERING
# =============================================================================

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates cycle-aware temporal and hormonal features.

    Feature groups created:
      - Rolling means of hormones (3-day, 7-day windows)
      - LH surge momentum (rate of change)
      - Symptom burden score (composite)
      - Days to next expected period (if cycle length estimable)
      - Lagged features for sequential modeling
    """
    df = df.copy().sort_values(["id", "day_in_study"]).reset_index(drop=True)

    # ---- Rolling Hormonal Features (per participant) ----
    for col in ["lh_imputed", "estrogen_imputed", "pdg_imputed"]:
        if col in df.columns:
            df[f"{col}_roll3"] = (
                df.groupby("id")[col]
                  .transform(lambda x: x.rolling(3, min_periods=1).mean())
            )
            df[f"{col}_roll7"] = (
                df.groupby("id")[col]
                  .transform(lambda x: x.rolling(7, min_periods=1).mean())
            )
            # Rate of change (momentum)
            df[f"{col}_delta"] = (
                df.groupby("id")[col]
                  .transform(lambda x: x.diff().fillna(0))
            )

    # ---- LH Surge Momentum (key ovulation signal) ----
    if "lh_imputed" in df.columns:
        df["lh_momentum"] = (
            df.groupby("id")["lh_imputed"]
              .transform(lambda x: x.pct_change().fillna(0).clip(-5, 5))
        )
        # Consecutive days of LH rise
        df["lh_rising"] = (df["lh_imputed_delta"] > 0).astype(int)
        df["lh_consecutive_rise"] = (
            df.groupby("id")["lh_rising"]
              .transform(lambda x: x * (x.groupby((x != x.shift()).cumsum()).cumcount() + 1))
        )

    # ---- Symptom Burden Score (composite severity) ----
    symptom_cols = [c for c in SYMPTOM_FEATURES if c in df.columns]
    if symptom_cols:
        df["symptom_burden"] = df[symptom_cols].sum(axis=1)
        df["symptom_burden_roll3"] = (
            df.groupby("id")["symptom_burden"]
              .transform(lambda x: x.rolling(3, min_periods=1).mean())
        )

    # ---- Cycle Phase Encoding (clinical phase boundaries) ----
    # phase: 1=menstrual, 2=follicular, 3=ovulatory, 4=luteal
    if "phase" in df.columns:
        df["is_luteal"]      = (df["phase"] == 4).astype(int)
        df["is_ovulatory"]   = (df["phase"] == 3).astype(int)
        df["is_follicular"]  = (df["phase"] == 2).astype(int)
        df["is_menstrual"]   = (df["phase"] == 1).astype(int)

    # ---- Lagged Features (prior day context) ----
    for col in ["cramps_imputed", "lh_imputed", "is_bleeding"]:
        if col in df.columns:
            df[f"{col}_lag1"] = df.groupby("id")[col].shift(1).fillna(0)
            df[f"{col}_lag2"] = df.groupby("id")[col].shift(2).fillna(0)

    # ---- Days Since Last Bleed ----
    if "is_bleeding" in df.columns:
        # Days since the most recent bleed day (uses past only; no look-ahead).
        bleed = df["is_bleeding"].astype(int)
        grp = bleed.groupby(df["id"]).cumsum()
        df["days_since_bleed"] = df.groupby(["id", grp]).cumcount()
        # Before the first bleed event in a participant, this value is undefined; keep 0.
        df.loc[grp == 0, "days_since_bleed"] = 0

    # ---- Estrogen:Progesterone Ratio (hormonal balance) ----
    if "estrogen_imputed" in df.columns and "pdg_imputed" in df.columns:
        denom = df["pdg_imputed"].replace(0, np.nan)
        df["e2_pdg_ratio"] = (df["estrogen_imputed"] / denom).fillna(0).clip(0, 100)

    print(f"[Feature Engineering] Created {len(df.columns)} total features from {len(df)} rows")
    return df


def get_engineered_feature_list(df: pd.DataFrame) -> list:
    """Returns full list of available features after engineering."""
    engineered = [
        c for c in df.columns
        if any(suffix in c for suffix in
               ["_roll3", "_roll7", "_delta", "_momentum", "_rise",
                "_burden", "is_luteal", "is_ovulatory", "is_follicular",
                "is_menstrual", "_lag1", "_lag2", "days_since_bleed",
                "e2_pdg_ratio", "consecutive_rise"])
    ]
    base = [c for c in ALL_FEATURES if c in df.columns]
    print("ordered features from engr feature list after enginnering")
    print(list(dict.fromkeys(base + engineered)))
    print("*" * 20)
    return list(dict.fromkeys(base + engineered))  # preserve order, no duplicates


In [8]:
"""
Prediction Validator Module
Validates input features before making predictions in the pipeline.
Integrates with the main pipeline to catch feature mismatches early.
"""
def validate_prediction_input(
    features: pd.DataFrame,
    model,
    verbose: bool = True,
    raise_on_mismatch: bool = True
) -> Dict[str, any]:
    """
    Comprehensive validation of features before prediction.
    
    Args:
        features: Input dataframe for prediction
        model: Trained sklearn model with feature_names_in_ attribute
        verbose: Print diagnostic output
        raise_on_mismatch: Raise error if features don't match model training
        
    Returns:
        Validation result dict with status and details
    """
    
    result = {
        "valid": True,
        "errors": [],
        "warnings": [],
        "diagnostics": {}
    }
    
    # ========== 1. SHAPE & INTEGRITY CHECKS ==========
    if verbose:
        print("\n" + "="*60)
        print("PREDICTION INPUT VALIDATION")
        print("="*60)
        print(f"\n[SHAPE] Features shape: {features.shape}")
    
    result["diagnostics"]["shape"] = features.shape
    
    # Check for NaNs
    nan_check = features.isnull().any().any()
    nan_by_col = features.isnull().sum()
    
    if verbose:
        print(f"[NaN CHECK] Any missing values: {nan_check}")
        if nan_check:
            print(f"  Columns with NaNs: {nan_by_col[nan_by_col > 0].to_dict()}")
    
    if nan_check:
        result["warnings"].append("Input contains NaN values - will cause prediction errors")
        result["diagnostics"]["nans_per_column"] = nan_by_col[nan_by_col > 0].to_dict()
    
    # ========== 2. FEATURE NAME & ORDER MATCHING ==========
    expected_features = list(model.feature_names_in_)
    actual_features = features.columns.tolist()
    
    if verbose:
        print(f"\n[FEATURES] Expected ({len(expected_features)}): {expected_features}")
        print(f"[FEATURES] Actual   ({len(actual_features)}): {actual_features}")
    
    # Check for exact match (name + order)
    features_match_exact = expected_features == actual_features
    
    if verbose:
        print(f"[MATCH] Exact match (name + order): {features_match_exact}")
    
    if not features_match_exact:
        result["valid"] = False
        result["errors"].append("Feature names or order don't match model training")
        
        # Detailed mismatch analysis
        missing = set(expected_features) - set(actual_features)
        extra = set(actual_features) - set(expected_features)
        
        if missing:
            result["errors"].append(f"  Missing features: {missing}")
            if verbose:
                print(f"  [ERROR] Missing features: {missing}")
        
        if extra:
            result["errors"].append(f"  Extra features: {extra}")
            if verbose:
                print(f"  [ERROR] Extra features: {extra}")
        
        # Check if it's just an ordering issue
        if set(expected_features) == set(actual_features):
            result["warnings"].append("Feature names match but order differs - reordering required")
            if verbose:
                print(f"  [WARNING] Features match but different order")
    
    # ========== 3. DATA TYPE & VALUE CHECKS ==========
    if verbose:
        print(f"\n[DATA TYPES]")
    
    for col in actual_features:
        dtype = features[col].dtype
        if verbose:
            print(f"  {col}: {dtype}")
        
        # Warn about object/string types (should be numeric for most models)
        if dtype == "object":
            result["warnings"].append(f"Column '{col}' is object type - may cause prediction issues")
    
    # ========== 4. VALUE RANGE CHECKS ==========
    if verbose:
        print(f"\n[VALUE RANGES]")
    
    for col in actual_features:
        col_min = features[col].min()
        col_max = features[col].max()
        col_mean = features[col].mean()
        
        if verbose:
            print(f"  {col}: min={col_min:.2f}, max={col_max:.2f}, mean={col_mean:.2f}")
        
        result["diagnostics"][f"{col}_stats"] = {
            "min": float(col_min),
            "max": float(col_max),
            "mean": float(col_mean),
            "std": float(features[col].std())
        }
    
    # ========== 5. SAMPLE DATA PREVIEW ==========
    if verbose:
        print(f"\n[SAMPLE DATA] First row (transposed for readability):")
        print(features.iloc[0].to_string())
    
    # ========== 6. FINAL DECISION ==========
    if verbose:
        print("\n" + "="*60)
        if result["valid"]:
            print("✓ VALIDATION PASSED - Safe to predict")
        else:
            print("✗ VALIDATION FAILED - Do not proceed with prediction")
            for error in result["errors"]:
                print(f"  ERROR: {error}")
        
        if result["warnings"]:
            print("\nWarnings:")
            for warning in result["warnings"]:
                print(f"  ⚠ {warning}")
        print("="*60 + "\n")
    
    if result["valid"] is False and raise_on_mismatch:
        raise ValueError(
            f"Feature validation failed:\n" + 
            "\n".join(result["errors"])
        )
    
    return result


def reorder_features_for_model(
    features: pd.DataFrame,
    model
) -> pd.DataFrame:
    """
    Reorder and select features to match model's expected input.
    Useful when feature names match but order differs.
    
    Args:
        features: Input dataframe
        model: Trained model with feature_names_in_
        
    Returns:
        Reordered dataframe matching model's expected features
    """
    expected_features = list(model.feature_names_in_)
    
    try:
        features_reordered = features[expected_features]
        print(f"✓ Features reordered to match model training")
        return features_reordered
    except KeyError as e:
        raise ValueError(
            f"Cannot reorder features - missing columns: {e}"
        )


def safe_predict(
    model,
    features: pd.DataFrame,
    validate: bool = True,
    verbose: bool = True
) -> np.ndarray:
    """
    Predict with built-in validation.
    
    Args:
        model: Trained sklearn model
        features: Input features
        validate: Run validation before predicting
        verbose: Print diagnostic output
        
    Returns:
        Prediction array
    """
    
    if validate:
        validation = validate_prediction_input(
            features, model, verbose=verbose, raise_on_mismatch=True
        )
    
    # Ensure feature order matches
    features_ordered = reorder_features_for_model(features, model)
    
    # Make prediction
    predictions = model.predict(features_ordered)
    
    if verbose:
        print(f"[PREDICTION] Shape: {predictions.shape}")
        print(f"[PREDICTION] Sample: {predictions[:5]}")
    
    return predictions


# ============================================================================
# Example integration into pipeline
# ============================================================================

def predict_with_validation_example(model, features: pd.DataFrame):
    """
    Example of how to integrate validation into the pipeline's prediction step.
    """
    
    # Method 1: Manual validation (more control)
    print("\n[STEP X] Validating prediction input...")
    validation = validate_prediction_input(
        features, 
        model, 
        verbose=True,
        raise_on_mismatch=True  # Fail fast if something is wrong
    )
    
    if validation["valid"]:
        features_clean = reorder_features_for_model(features, model)
        predictions = model.predict(features_clean)
        probabilities = model.predict_proba(features_clean)
        return predictions, probabilities
    
    # Method 2: All-in-one (simpler)
    # predictions = safe_predict(model, features, validate=True, verbose=True)
    # return predictions

In [9]:
# =============================================================================
# SECTION 4 (FIXED): MODEL DEFINITIONS
# =============================================================================

def build_models(y_train: pd.Series = None):
    """
    Returns dict of models to train and evaluate.
    
    All models handle class imbalance natively — NO SMOTE needed.
    
    Args:
        y_train: Training labels. If provided, computes scale_pos_weight 
                 for LightGBM automatically. If None, uses is_unbalance=True.
    
    Models:
      - LogReg:       Linear baseline, coefficient-based explainability
      - RandomForest: MDI feature importance, SHAP TreeExplainer compatible
      - LightGBM:     High performance, replaces GradientBoosting,
                      natively supports imbalanced data via scale_pos_weight
    """
    
    # Compute scale_pos_weight for LightGBM
    # = ratio of negative to positive samples
    # e.g. if 1 period day per 28 days → scale = 27
    if y_train is not None:
        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        scale_pos_weight = neg / pos if pos > 0 else 10
        print(f"[LightGBM] scale_pos_weight = {scale_pos_weight:.2f} "
              f"(neg={neg}, pos={pos})")
    else:
        scale_pos_weight = None  # fallback: use is_unbalance=True

    models = {

        # ------------------------------------------------------------------
        # 1. Logistic Regression (Baseline)
        # class_weight="balanced" automatically adjusts weights inversely
        # proportional to class frequencies → no resampling needed
        # ------------------------------------------------------------------
        "Logistic Regression (Baseline)": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",   # ← handles imbalance internally
                C=0.1,
                solver="lbfgs",
                random_state=42
            ))
        ]),

        # ------------------------------------------------------------------
        # 2. Random Forest
        # class_weight="balanced_subsample" applies balanced weighting
        # per tree (better than "balanced" for forests)
        # ------------------------------------------------------------------
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=5,
            max_features="sqrt",
            class_weight="balanced_subsample",  # ← better than "balanced" for RF
            random_state=42,
            n_jobs=-1
        ),

        # ------------------------------------------------------------------
        # 3. LightGBM (replaces GradientBoostingClassifier)
        # scale_pos_weight explicitly tells the model the minority class
        # is rare — directly optimized for imbalanced binary classification
        # ------------------------------------------------------------------
        "LightGBM": LGBMClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_samples=10,
            
            # Imbalance handling:
            # Use scale_pos_weight if computed, else use is_unbalance flag
            scale_pos_weight=scale_pos_weight if scale_pos_weight else 1,
            is_unbalance=True if scale_pos_weight is None else False,
            
            random_state=42,
            n_jobs=-1,
            verbose=-1          # suppress LightGBM's verbose output
        ),
    }

    return models


# =============================================================================
# SECTION 4.5 (FIXED): CLASS IMBALANCE HANDLER
# =============================================================================

def handle_class_imbalance(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    strategy: str = "class_weight"
):
    """
    Handles class imbalance without resampling temporal/lag features.
    
    Args:
        X_train:  Training features (may contain lag/rolling features)
        y_train:  Training labels
        strategy: One of:
                    "class_weight"  → recommended, no resampling (default)
                    "oversample"    → simple minority oversampling (safe for lags)
                    "smote_safe"    → SMOTE on non-temporal features only
    
    Returns:
        X_train, y_train (possibly resampled depending on strategy)
    
    WHY NOT SMOTE on full feature set:
        Lag features (lh_lag1, pdg_lag2, rolling_mean_7 etc.) are 
        time-dependent. SMOTE interpolates between rows, creating synthetic
        rows where lag1 doesn't correspond to the actual previous day.
        This introduces unrealistic training samples and can mislead the model.
    """
    
    pos_rate = y_train.mean()
    neg      = (y_train == 0).sum()
    pos      = (y_train == 1).sum()
    
    print(f"\n[Class Balance]")
    print(f"  Total samples:  {len(y_train)}")
    print(f"  Positive (1):   {pos}  ({pos_rate:.3f})")
    print(f"  Negative (0):   {neg}  ({1-pos_rate:.3f})")
    print(f"  Strategy:       {strategy}")

    # -----------------------------------------------------------------------
    # STRATEGY 1: class_weight (RECOMMENDED)
    # No resampling — models handle imbalance via weighted loss
    # Works with: LogisticRegression, RandomForest, LightGBM
    # -----------------------------------------------------------------------
    if strategy == "class_weight":
        print("  → No resampling. Models use class_weight/scale_pos_weight internally.")
        print("  → Temporal features (lags, rolling) are preserved exactly.")
        return X_train, y_train

    # -----------------------------------------------------------------------
    # STRATEGY 2: Simple Oversampling (safe for temporal features)
    # Repeats real minority rows — doesn't interpolate lag features
    # Less powerful than SMOTE but temporally safe
    # -----------------------------------------------------------------------
    elif strategy == "oversample":
        if pos_rate < 0.15:
            minority_X = X_train[y_train == 1]
            minority_y = y_train[y_train == 1]
            
            # How many times to repeat minority class
            repeat_factor = int((neg / pos) * 0.5)  # partial oversampling
            
            X_minority_rep = pd.concat([minority_X] * repeat_factor, ignore_index=True)
            y_minority_rep = pd.concat([minority_y] * repeat_factor, ignore_index=True)
            
            X_bal = pd.concat([X_train, X_minority_rep], ignore_index=True)
            y_bal = pd.concat([y_train, y_minority_rep], ignore_index=True)
            
            print(f"  → After oversampling: {len(X_bal)} samples | "
                  f"Positive rate: {y_bal.mean():.3f}")
            return X_bal, y_bal
        else:
            print("  → Balance acceptable, no oversampling needed.")
            return X_train, y_train

    # -----------------------------------------------------------------------
    # STRATEGY 3: SMOTE on non-temporal features only
    # Applies SMOTE to hormone/symptom features, repeats real rows for lags
    # More complex but gives synthetic diversity without corrupting lags
    # -----------------------------------------------------------------------
    elif strategy == "smote_safe":
        
        if pos_rate >= 0.15:
            print("  → Balance acceptable, skipping SMOTE.")
            return X_train, y_train
        
        # Separate temporal vs non-temporal features
        temporal_keywords = ["lag1", "lag2", "rolling", "momentum", "surge"]
        temporal_cols     = [c for c in X_train.columns 
                             if any(kw in c for kw in temporal_keywords)]
        nontemporal_cols  = [c for c in X_train.columns 
                             if c not in temporal_cols]
        
        print(f"  → Temporal cols (no SMOTE): {len(temporal_cols)}")
        print(f"  → Non-temporal cols (SMOTE): {len(nontemporal_cols)}")
        
        # SMOTE on non-temporal features
        k = min(5, pos - 1)
        sm = SMOTE(random_state=42, k_neighbors=k)
        X_nontemp_bal, y_bal = sm.fit_resample(
            X_train[nontemporal_cols], y_train
        )
        
        # For temporal features: repeat real minority rows to match synthetic count
        n_synthetic    = len(X_nontemp_bal) - len(X_train)
        minority_temp  = X_train[temporal_cols][y_train == 1]
        extra_temporal = minority_temp.sample(
            n=n_synthetic, replace=True, random_state=42
        )
        
        X_temporal_bal = pd.concat(
            [X_train[temporal_cols], extra_temporal], ignore_index=True
        )
        
        # Recombine non-temporal (SMOTEd) + temporal (repeated real rows)
        X_bal = pd.concat([
            pd.DataFrame(X_nontemp_bal, columns=nontemporal_cols),
            X_temporal_bal.reset_index(drop=True)
        ], axis=1)[X_train.columns]  # restore original column order
        
        print(f"  → After safe SMOTE: {len(X_bal)} samples | "
              f"Positive rate: {pd.Series(y_bal).mean():.3f}")
        
        return X_bal, pd.Series(y_bal)

    else:
        raise ValueError(
            f"Unknown strategy '{strategy}'. "
            f"Choose from: 'class_weight', 'oversample', 'smote_safe'"
        )

In [10]:
# =============================================================================
# SECTION 5: TRAINING & EVALUATION
# =============================================================================

def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.Series,
                   model_name: str) -> dict:
    """Computes clinical-grade evaluation metrics."""
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else y_pred

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    auc    = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
    auprc  = average_precision_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
    cm     = confusion_matrix(y_test, y_pred)

    metrics = {
        "model":     model_name,
        "accuracy":  report["accuracy"],
        "precision": report.get("1", {}).get("precision", 0),
        "recall":    report.get("1", {}).get("recall", 0),
        "f1_score":  report.get("1", {}).get("f1-score", 0),
        "auc_roc":   auc,
        "auprc":     auprc,
        "confusion_matrix": cm.tolist()
    }

    print(f"\n  {'='*50}")
    print(f"  Model: {model_name}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1_score']:.4f}")
    print(f"  AUC-ROC:   {metrics['auc_roc']:.4f}")
    print(f"  AUPRC:     {metrics['auprc']:.4f}")
    print(f"  Confusion Matrix:\n{cm}")

    return metrics


def cross_validate_model(model, X: pd.DataFrame, y: pd.Series,
                         model_name: str, n_splits: int = 5) -> dict:
    """Stratified k-fold cross-validation for robust performance estimates."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    cv_auc = cross_val_score(model, X, y, cv=skf, scoring="roc_auc", n_jobs=-1)
    cv_f1  = cross_val_score(model, X, y, cv=skf, scoring="f1", n_jobs=-1)

    cv_results = {
        "model":       model_name,
        "cv_auc_mean": cv_auc.mean(),
        "cv_auc_std":  cv_auc.std(),
        "cv_f1_mean":  cv_f1.mean(),
        "cv_f1_std":   cv_f1.std(),
    }

    print(f"\n  [CV] {model_name}")
    print(f"  AUC-ROC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
    print(f"  F1:      {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}")

    return cv_results

In [11]:
# =============================================================================
# SECTION 6: EXPLAINABILITY — FEATURE IMPORTANCE
# =============================================================================

def extract_feature_importance(model, feature_names: list,
                                model_name: str) -> pd.DataFrame:
    """
    Extracts feature importances using model-native methods.

    For tree-based models: Mean Decrease in Impurity (MDI).
    For linear models: Absolute coefficient values.

    Note: When SHAP is available (pip install shap), replace MDI with:
        import shap
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)
        This provides individual prediction explanations (local) as well as
        global feature importance, suitable for clinical DSS output.
    """
    importance_df = None

    if hasattr(model, "feature_importances_"):
        # Tree-based: MDI importance
        importances = model.feature_importances_
        importance_df = pd.DataFrame({
            "feature":    feature_names,
            "importance": importances,
            "method":     "MDI (Tree-based)"
        }).sort_values("importance", ascending=False)

    elif hasattr(model, "named_steps"):
        # Pipeline (Logistic Regression)
        clf = model.named_steps.get("clf")
        if hasattr(clf, "coef_"):
            coef = np.abs(clf.coef_[0])
            importance_df = pd.DataFrame({
                "feature":    feature_names,
                "importance": coef,
                "method":     "Absolute Coefficient"
            }).sort_values("importance", ascending=False)

    if importance_df is not None:
        print(f"\n[Explainability] Top 15 Features — {model_name}")
        print(importance_df.head(15).to_string(index=False))

    return importance_df


def generate_clinical_explanation(prediction_proba: float,
                                   top_features: pd.DataFrame,
                                   target: str = "period_start") -> str:
    """
    Generates a plain-language clinical explanation for a prediction.
    This is the interface layer for the RAG/LLM agent — it provides
    structured context that the agent augments with medical literature.

    In the full Agentic RAG system:
      1. This function provides the factual prediction context
      2. The agent retrieves relevant clinical guidelines (e.g., ACOG, FIGO)
      3. The LLM synthesizes an evidence-grounded explanation
    """
    risk_level = "High" if prediction_proba > 0.7 else "Moderate" if prediction_proba > 0.4 else "Low"
    top_3 = top_features.head(3)["feature"].tolist() if top_features is not None else []

    explanation = (
        f"PREDICTION SUMMARY\n"
        f"{'='*40}\n"
        f"Target Event:    {'Period Onset' if target == 'period_start' else 'LH Surge / Ovulation'}\n"
        f"Probability:     {prediction_proba:.1%}\n"
        f"Risk Level:      {risk_level}\n"
        f"\nKEY CONTRIBUTING FACTORS\n"
        f"{'-'*40}\n"
    )
    for i, feat in enumerate(top_3, 1):
        explanation += f"  {i}. {feat.replace('_', ' ').title()}\n"

    explanation += (
        f"\nINTERPRETATION\n"
        f"{'-'*40}\n"
        f"The model identifies {risk_level.lower()} probability of "
        f"{'menstruation onset' if target == 'period_start' else 'ovulation'} "
        f"based on the hormonal and symptomatic patterns observed. "
        f"[RAG Agent will augment this with clinical literature context.]\n"
    )
    return explanation

In [12]:
# =============================================================================
# SECTION 7: AGENTIC RAG INTEGRATION STUB
# =============================================================================

class AgenticRAGStub:
    """
    Stub for the Agentic RAG module.

    Full Implementation Architecture:
    ----------------------------------
    1. Vector Store:
       - Embed medical literature (PubMed abstracts, ACOG/FIGO guidelines,
         menstrual health clinical trials) using sentence-transformers
       - Store in FAISS or ChromaDB with metadata (source, date, evidence level)

    2. Retrieval Agent:
       - On each prediction, query vector store with:
           query = f"menstrual period onset prediction {top_features}"
       - Retrieve top-k (k=5) relevant passages with evidence grades

    3. LLM Synthesis (Claude / GPT-4):
       - System prompt: clinical DSS assistant with explainability focus
       - Input: SHAP explanation + retrieved passages + patient cycle summary
       - Output: Structured clinical explanation with citations

    4. Explainability Audit Trail:
       - Log prediction, SHAP values, retrieved docs, and LLM output
       - Human-in-the-loop review interface for clinicians

    Sample RAG Query Construction:
    --------------------------------
    rag_query = {
        "prediction": prediction_proba,
        "key_features": top_features[:5].to_dict(),
        "cycle_day": day_in_period,
        "phase": phase,
        "retrieval_query": "luteal phase period onset hormonal markers clinical"
    }
    """

    def __init__(self):
        self.vector_store = None  # Replace: FAISS.load_local("medical_index")
        self.llm = None           # Replace: Anthropic() or OpenAI()

    def retrieve_and_explain(self, prediction_context: dict) -> str:
        """
        Placeholder: returns structured context for LLM augmentation.
        In production, this queries the vector store and calls the LLM.
        """
        return (
            "[RAG MODULE — Integration Point]\n"
            "In the full system, this module:\n"
            "  1. Embeds prediction context → queries medical vector store\n"
            "  2. Retrieves top-5 relevant clinical passages\n"
            "  3. Sends to LLM for evidence-grounded explanation\n"
            "  4. Returns clinician-readable report with citations\n\n"
            f"Context received: {json.dumps(prediction_context, indent=2)}"
        )


In [13]:
def run_pipeline_with_validation(df, output_dir="."):
    """
    Enhanced version of run_pipeline with built-in validation.
    Drop-in replacement for the original run_pipeline.

    Returns:
        all_results  : dict of metrics/cv/feature importance per target
        best_models  : dict keyed by target, each containing:
                         'model'      → trained model object
                         'model_name' → name string
                         'auc'        → best AUC-ROC score
                         'X_train'    → balanced training features
                         'y_train'    → balanced training labels
    """

    os.makedirs(output_dir, exist_ok=True)
    all_results = {}
    best_models = {}   # ← FIX: per-target dict instead of single variable

    print("\n" + "="*60)
    print("  MENSTRUAL CYCLE PREDICTION PIPELINE")
    print("  Explainable AI Healthcare Decision Support System")
    print("="*60)

    # Step 1: Feature Engineering
    print("\n[STEP 1] Feature Engineering...")
    df_eng       = engineer_features(df)
    feature_cols = get_engineered_feature_list(df_eng)

    # Step 2: Determine targets
    targets = [TARGET_PERIOD]
    if TARGET_OVUL in df_eng.columns:
        targets.append(TARGET_OVUL)

    for target in targets:
        if target not in df_eng.columns:
            print(f"\n[SKIP] Target '{target}' not found in data.")
            continue

        print(f"\n{'='*60}")
        print(f"  TARGET: {target.upper()}")
        print(f"{'='*60}")

        # Step 3: Data Preparation
        X_train, X_test, y_train, y_test, used_features = temporal_train_test_split(
            df_eng, target, feature_cols
        )

        # Step 4: Class Imbalance
        X_train_bal, y_train_bal = handle_class_imbalance(
            X_train, y_train, strategy="class_weight"
        )

        # Step 5: Train & Evaluate Models
        models         = build_models(y_train=y_train_bal)
        target_results = {"metrics": [], "cv_results": [], "feature_importance": {}}

        # ← FIX: scoped per target iteration (not leaking across targets)
        best_model      = None
        best_auc        = 0
        best_model_name = ""

        for model_name, model in models.items():
            print(f"\n[TRAINING] {model_name}")
            model.fit(X_train_bal, y_train_bal)

            # Holdout evaluation
            metrics = evaluate_model(model, X_test, y_test, model_name)
            target_results["metrics"].append(metrics)

            # Cross-validation
            cv_res = cross_validate_model(model, X_train, y_train, model_name)
            target_results["cv_results"].append(cv_res)

            # Feature Importance
            fi_df = extract_feature_importance(model, used_features, model_name)
            if fi_df is not None:
                target_results["feature_importance"][model_name] = fi_df

            # Track best model by AUC
            if metrics["auc_roc"] > best_auc:
                best_auc        = metrics["auc_roc"]
                best_model      = model
                best_model_name = model_name

        # Step 6: Clinical Explanation for Best Model
        print(f"\n[BEST MODEL] {best_model_name} (AUC-ROC: {best_auc:.4f})")
        best_fi = target_results["feature_importance"].get(best_model_name)

        # Step 6.5: Input Validation
        print(f"\n[STEP 6.5] Input Validation")
        sample_features = X_test.iloc[0:1]

        # ← FIX: removed hardcoded 0.72 fallback — raises cleanly instead
        try:
            validation = validate_prediction_input(
                sample_features,
                best_model,
                verbose=True,
                raise_on_mismatch=True
            )
            sample_features_clean = reorder_features_for_model(sample_features, best_model)
            sample_proba          = best_model.predict_proba(sample_features_clean)[0, 1]

        except ValueError as e:
            print(f"[ERROR] Validation failed: {e}")
            print("[WARNING] Skipping clinical explanation for this target.")
            sample_proba = None   # ← FIX: None instead of 0.72

        # Only generate explanation if prediction succeeded
        if sample_proba is not None:
            explanation = generate_clinical_explanation(sample_proba, best_fi, target)
            print("\n" + explanation)

            # Step 7: RAG Stub
            rag = AgenticRAGStub()
            rag_context = {
                "target":       target,
                "top_model":    best_model_name,
                "auc_roc":      best_auc,
                "prediction":   sample_proba,
                "top_features": best_fi.head(5)["feature"].tolist() if best_fi is not None else []
            }
            rag_output = rag.retrieve_and_explain(rag_context)
            print("\n" + rag_output)
        else:
            print("[SKIP] RAG explanation skipped due to validation failure.")

        # ← FIX: store best model per target with all context needed later
        best_models[target] = {
            "model":      best_model,
            "model_name": best_model_name,
            "auc":        best_auc,
            "X_train":    X_train_bal,
            "y_train":    y_train_bal,
        }

        # Save results
        all_results[target] = target_results
        results_path = os.path.join(output_dir, f"results_{target}.json")
        serializable = {
            "metrics":    target_results["metrics"],
            "cv_results": target_results["cv_results"],
            "best_model": best_model_name,
            "best_auc":   best_auc,
            "feature_importance_top20": {
                k: v.head(20).to_dict(orient="records")
                for k, v in target_results["feature_importance"].items()
            }
        }
        with open(results_path, "w") as f:
            json.dump(serializable, f, indent=2)
        print(f"\n[SAVED] Results → {results_path}")

    # ← FIX: return best_models dict instead of single leaked variable
    return all_results, best_models


In [14]:
# Drop these before calling run_pipeline()
leaky_features = [
    # Directly derived from target
    "is_menstrual",
    "is_bleeding",
    "is_bleeding_lag1",
    "is_bleeding_lag2",
    "phase",
    "is_luteal",
    "is_follicular",
    "is_ovulatory",
    "day_in_period",
    # Flow = bleeding = period
    "flow_volume_imputed",
    "flow_grp_Abnormal",
    "flow_grp_Fresh",
    "flow_grp_Old",
    "flow_grp_No Flow",
    # Year column — not clinically meaningful
    "study_interval",
    # Raw un-imputed columns — have NaNs, imputed versions already in ALL_FEATURES
    "lh", "estrogen", "pdg", "cramps", "sorebreasts",
    "bloating", "moodswing", "fatigue", "headaches",
    "foodcravings", "indigestion", "exerciselevel",
    "stress", "sleepissue", "appetite",
    # Intermediate columns
    "estrogen_clean",
    "flow_volume_numeric",
    "flow_volume",
    "flow_color",
    "days_since_bleed",   # derived from is_bleeding → leaky
]

df_model = df.drop(columns=[c for c in leaky_features if c in df.columns])

In [15]:
# Pick 2-3 participants to NEVER touch during training
# Choose them before running the pipeline — fix them permanently
HOLDOUT_IDS = [50, 9, 2]  # replace with actual IDs from your data

# Verify they have period start days
print(df_model[df_model["id"].isin(HOLDOUT_IDS)]["period_start"].value_counts())

period_start
False    455
True      18
Name: count, dtype: int64


In [16]:
# Separate holdout BEFORE pipeline runs
df_holdout = df_model[df_model["id"].isin(HOLDOUT_IDS)].copy()
df_train   = df_model[~df_model["id"].isin(HOLDOUT_IDS)].copy()
print(f"Training data:  {len(df_train)} rows, {df_train['id'].nunique()} participants")
print(f"Holdout data:   {len(df_holdout)} rows, {df_holdout['id'].nunique()} participants")
print(f"Holdout period start days: {df_holdout['period_start'].sum()}")

all_results, best_models = run_pipeline_with_validation(df_train, output_dir="./pipeline_outputs")

period_model      = best_models["period_start"]["model"]
period_model_name = best_models["period_start"]["model_name"]
period_auc        = best_models["period_start"]["auc"]

if "lh_surge" in best_models:
    ovulation_model = best_models["lh_surge"]["model"]

# Load & engineer unseen participant
test_df         = pd.read_csv(r"D:\MajorProject\prediction_pipeline\unseen_participant.csv")
test_engineered = engineer_features(test_df)

# ✅ FIX: use exact training features — no mismatch possible
trained_features = list(period_model.feature_names_in_)
last_row         = test_engineered.tail(1).reindex(columns=trained_features, fill_value=0)
print(last_row[["pdg_imputed", "pdg_imputed_delta", "cramps_imputed_lag1", "e2_pdg_ratio"]].T)

proba = period_model.predict_proba(last_row)[:, 1][0]
pred  = period_model.predict(last_row)[0]

print(f"Prediction:  {'Period Start' if pred == 1 else 'No Period'}")
print(f"Probability: {proba:.1%}")

Training data:  5186 rows, 39 participants
Holdout data:   473 rows, 3 participants
Holdout period start days: 18

  MENSTRUAL CYCLE PREDICTION PIPELINE
  Explainable AI Healthcare Decision Support System

[STEP 1] Feature Engineering...
[Feature Engineering] Created 42 total features from 5186 rows
ordered features from engr feature list after enginnering
['lh_imputed', 'estrogen_imputed', 'pdg_imputed', 'high_estrogen_flag', 'estrogen_capped_flag', 'day_in_study', 'is_weekend', 'cramps_imputed', 'sorebreasts_imputed', 'bloating_imputed', 'moodswing_imputed', 'fatigue_imputed', 'headaches_imputed', 'foodcravings_imputed', 'indigestion_imputed', 'exerciselevel_imputed', 'stress_imputed', 'sleepissue_imputed', 'appetite_imputed', 'lh_imputed_roll3', 'lh_imputed_roll7', 'lh_imputed_delta', 'estrogen_imputed_roll3', 'estrogen_imputed_roll7', 'estrogen_imputed_delta', 'pdg_imputed_roll3', 'pdg_imputed_roll7', 'pdg_imputed_delta', 'lh_momentum', 'lh_consecutive_rise', 'symptom_burden', 'sym

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\MajorProject\\prediction_pipeline\\unseen_participant.csv'

In [17]:
def validate_on_holdout(participant_id, df_holdout, model, model_name="period_start"):
    """
    Tests model on a completely unseen participant.
    Shows prediction vs ground truth for every day.
    """
    participant_data      = df_holdout[df_holdout["id"] == participant_id].copy()
    participant_engineered = engineer_features(participant_data)
    
    trained_features = list(model.feature_names_in_)
    
    results = []
    for idx in range(len(participant_engineered)):
        row = participant_engineered.iloc[[idx]].reindex(
            columns=trained_features, fill_value=0
        )
        proba  = model.predict_proba(row)[:, 1][0]
        pred   = model.predict(row)[0]
        actual = participant_engineered.iloc[idx]["period_start"]
        
        results.append({
            "day":        participant_engineered.iloc[idx]["day_in_study"],
            "actual":     int(actual),
            "predicted":  int(pred),
            "probability": round(proba, 4),
            "correct":    int(pred) == int(actual)
        })
    
    results_df = pd.DataFrame(results)
    
    # Summary
    period_days = results_df[results_df["actual"] == 1]
    non_period  = results_df[results_df["actual"] == 0]
    
    print(f"\n{'='*50}")
    print(f"Holdout Participant: {participant_id}")
    print(f"{'='*50}")
    print(f"Total days:          {len(results_df)}")
    print(f"Period start days:   {len(period_days)}")
    print(f"Overall accuracy:    {results_df['correct'].mean():.1%}")
    
    print(f"\n--- Period Start Days (should be high probability) ---")
    print(period_days[["day", "actual", "predicted", "probability"]].to_string(index=False))
    
    print(f"\n--- Sample Non-Period Days (should be low probability) ---")
    print(non_period[["day", "actual", "predicted", "probability"]].sample(5, random_state=42).to_string(index=False))
    
    return results_df


# Run for all holdout participants
for pid in HOLDOUT_IDS:
    validate_on_holdout(pid, df_holdout, period_model)

[Feature Engineering] Created 42 total features from 193 rows

Holdout Participant: 50
Total days:          193
Period start days:   9
Overall accuracy:    99.0%

--- Period Start Days (should be high probability) ---
 day  actual  predicted  probability
   6       1          1       0.9972
  27       1          1       0.9970
  50       1          1       0.9998
  63       1          0       0.0740
  88       1          1       0.9998
 850       1          0       0.0111
 886       1          1       1.0000
 916       1          1       0.9999
 950       1          1       1.0000

--- Sample Non-Period Days (should be low probability) ---
 day  actual  predicted  probability
  21       0          0          0.0
  45       0          0          0.0
 923       0          0          0.0
 876       0          0          0.0
 914       0          0          0.0
[Feature Engineering] Created 42 total features from 190 rows

Holdout Participant: 9
Total days:          190
Period start days: 